## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# import os
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["OPENAI_BASE_URL"] = "https://ai.api.cloud.yandex.net/v1"


# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

# MODEL_NAME = "gpt://b1gmsujt9mgtp0qubblu/gpt-oss-20b/latest"
# os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
# llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Search for products in the catalog by keyword, category, brand, price, or sorting.

    Args:
        query: Free-text search query. Words are matched against product name, category, brand, and tags.
        category: Filter by product category. One of: headphones, earbuds, mouse, keyboard, ereader.
        brand: Filter by brand name (case-insensitive). Examples: Sony, Bose, Apple, Anker, Logitech, Keychron, NuPhy, Amazon.
        max_price: Maximum price in dollars. Only products at or below this price are returned.
        sort_by: Sort the results. Options: "price_asc" (cheapest first), "rating_desc" (highest rating first).

    Returns:
        A list of matching product dicts with keys: id, name, category, brand, price, color, rating, tags.
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Add a product to the shopping cart by its product ID.

    Args:
        product_id: The unique product identifier (e.g. "p1", "p2", ..., "p10").
        quantity: Number of units to add. Defaults to 1.

    Returns:
        A dict with "ok" (bool) indicating success, and "cart_size" or "error".
    """
    ...

SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    system_msg = SystemMessage(content=(
        "You are a helpful shopping assistant for an electronics store. "
        "You help users search for products and add them to their cart. "
        "Use the provided tools to search for products and add them to the cart. "
        "When asked to find the cheapest or best product, search first, then pick the right one."
    ))
    messages = [system_msg, HumanMessage(content=user_message)]

    max_iterations = 15
    for _ in range(max_iterations):
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content

        for tc in ai_msg.tool_calls:
            name = tc["name"]
            args = tc["args"]
            call_id = tc["id"]

            if name == "search_products":
                result = tools.search_products(**args)
                state.last_results = result
                tracer.record(name, args, result)
            elif name == "add_to_cart":
                result = tools.add_to_cart(state, **args)
                tracer.record(name, args, result)
            else:
                result = {"error": f"Unknown tool: {name}"}

            messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id))

    return ai_msg.content if ai_msg.content else "I could not complete the request."


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)


def update_profile(key: str, value: str) -> dict:
    """Save or update a user preference in the user profile.

    Args:
        key: The preference key to update. Recommended keys: name, brand, max_price, color, category.
        value: The value to set for the given key.

    Returns:
        A dict confirming the update with the key and value saved.
    """
    ...

SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
    convert_to_openai_tool(update_profile),
]


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.
    Returns (response: str, updated_history: list).
    """
    profile = load_profile(profile_path)

    system_content = (
        "You are a helpful shopping assistant for an electronics store with memory. "
        "You help users search for products and add them to their cart. "
        "Use the provided tools to search for products and add them to the cart. "
        "When the user mentions preferences (name, brand, budget/max_price, color, category), "
        "use the update_profile tool to save each preference immediately. "
    )
    if profile:
        system_content += f"\n\nUser profile (saved preferences): {json.dumps(profile, ensure_ascii=False)}"

    system_msg = SystemMessage(content=system_content)
    human_msg = HumanMessage(content=user_message)
    history.append(human_msg)

    messages = [system_msg] + history

    max_iterations = 15
    for _ in range(max_iterations):
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(ai_msg)
        history.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content, history

        for tc in ai_msg.tool_calls:
            name = tc["name"]
            args = tc["args"]
            call_id = tc["id"]

            if name == "search_products":
                result = tools.search_products(**args)
                state.last_results = result
                tracer.record(name, args, result)
            elif name == "add_to_cart":
                result = tools.add_to_cart(state, **args)
                tracer.record(name, args, result)
            elif name == "update_profile":
                profile = load_profile(profile_path)
                profile[args["key"]] = args["value"]
                save_profile(profile, profile_path)
                result = {"ok": True, "key": args["key"], "value": args["value"]}
                tracer.record(name, args, result)
            else:
                result = {"error": f"Unknown tool: {name}"}

            tool_msg = ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id)
            messages.append(tool_msg)
            history.append(tool_msg)

    response = ai_msg.content if ai_msg.content else "I could not complete the request."
    return response, history


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import re

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


SEARCH_ONLY_SCHEMA = [
    convert_to_openai_tool(search_products),
]


def _extract_max_price(text: str) -> float | None:
    """Extract max price from user query using multiple patterns."""
    text_lower = text.lower().replace("$", "").replace(",", "")
    patterns = [
        r'(?:under|below|less than|up to|no more than|within|cheaper than|max|maximum|budget)\s+(\d+(?:\.\d+)?)',
        r'(\d+(?:\.\d+)?)\s+(?:dollars|dollar|usd|bucks)\s+(?:or less|max|maximum|budget)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text_lower)
        if match:
            return float(match.group(1))
    return None


def _wants_add_to_cart(text: str) -> bool:
    """Check if the user wants to add a product to cart."""
    text_lower = text.lower()
    cart_patterns = [
        r'\badd\b.*\b(?:cart|basket)\b',
        r'\b(?:cart|basket)\b.*\badd\b',
        r'\bput\b.*\b(?:in|into)\b.*\b(?:cart|basket)\b',
        r'\band\s+add\s+(?:it|them|the|this|that)\b',
        r'\bbuy\s+(?:it|them|this|that)\b',
        r'\bpurchase\s+(?:it|them|this|that)\b',
        r'\b(?:buy|purchase)\b.*\bfor\s+me\b',
        r'\bi\s+(?:want|need)\s+to\s+(?:buy|purchase)\b',
        r'\b(?:add|put)\s+(?:it|them|this|that)\s+(?:in|into|to)\b',
        r'\band\s+(?:buy|purchase)\s+(?:it|them|the|this|that)\b',
    ]
    for pattern in cart_patterns:
        if re.search(pattern, text_lower):
            return True
    return False


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        # Extract max_price from query
        extracted_price = _extract_max_price(ctx.query)
        if extracted_price is not None:
            ctx.max_price = extracted_price

        system_msg = SystemMessage(content=(
            "You are a product retriever agent. Your job is to search the catalog for relevant products. "
            "Use the search_products tool to find up to 5 relevant products based on the user query. "
            "If the user mentions a maximum price, use the max_price parameter. "
            "Search with appropriate filters for category, brand, and price."
        ))
        messages = [system_msg, HumanMessage(content=ctx.query)]

        max_iterations = 10
        for _ in range(max_iterations):
            ai_msg = llm_chat(messages, tools=SEARCH_ONLY_SCHEMA)
            messages.append(ai_msg)

            if not ai_msg.tool_calls:
                break

            for tc in ai_msg.tool_calls:
                name = tc["name"]
                args = tc["args"]
                call_id = tc["id"]

                if name == "search_products":
                    # Also capture max_price from LLM's tool call if not extracted from query
                    if ctx.max_price is None and args.get("max_price") is not None:
                        ctx.max_price = float(args["max_price"])
                    result = tools.search_products(**args)
                    state.last_results = result
                    tracer.record(name, args, result)
                else:
                    result = {"error": f"Unknown tool: {name}"}

                messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id))

        ctx.candidates = state.last_results[:5] if state.last_results else []
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        if not ctx.candidates:
            return ctx

        products_desc = json.dumps(
            [{k: p[k] for k in ("id", "name", "brand", "price", "rating", "tags") if k in p}
             for p in ctx.candidates],
            ensure_ascii=False
        )

        system_msg = SystemMessage(content=(
            "You are a product analyst specializing in advantages. "
            "For each product, write exactly 1-2 sentences about its PROS (advantages). "
            "You MUST respond in this exact format, one product per line:\n"
            "<product_id>: <pros text>\n\n"
            "Example:\n"
            "p1: Excellent noise cancellation and premium build quality.\n"
            "p2: Great value for the price with solid features.\n\n"
            "Use the exact product IDs provided. Do not add any other text."
        ))
        human_msg = HumanMessage(content=f"Analyze pros for these products:\n{products_desc}")

        ai_msg = llm_chat([system_msg, human_msg])

        for candidate in ctx.candidates:
            pid = candidate["id"]
            for line in ai_msg.content.split("\n"):
                line = line.strip()
                if not line:
                    continue
                pattern = rf'(?:^[-*\s]*)?{re.escape(pid)}\s*[:\-–—]\s*(.+)'
                match = re.search(pattern, line, re.IGNORECASE)
                if match:
                    ctx.pros[pid] = match.group(1).strip()
                    break
            if pid not in ctx.pros:
                ctx.pros[pid] = f"Rated {candidate['rating']}/5 with features: {', '.join(candidate.get('tags', []))}"

        tracer.record("analyze_pros", {"candidates": [c["id"] for c in ctx.candidates]}, ctx.pros)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        if not ctx.candidates:
            return ctx

        products_desc = json.dumps(
            [{k: p[k] for k in ("id", "name", "brand", "price", "rating", "tags") if k in p}
             for p in ctx.candidates],
            ensure_ascii=False
        )

        system_msg = SystemMessage(content=(
            "You are a product analyst specializing in disadvantages. "
            "For each product, write exactly 1-2 sentences about its CONS (disadvantages). "
            "You MUST respond in this exact format, one product per line:\n"
            "<product_id>: <cons text>\n\n"
            "Example:\n"
            "p1: Premium price may be too high for budget-conscious buyers.\n"
            "p2: Build quality is not as durable as higher-end models.\n\n"
            "Use the exact product IDs provided. Do not add any other text."
        ))
        human_msg = HumanMessage(content=f"Analyze cons for these products:\n{products_desc}")

        ai_msg = llm_chat([system_msg, human_msg])

        for candidate in ctx.candidates:
            pid = candidate["id"]
            for line in ai_msg.content.split("\n"):
                line = line.strip()
                if not line:
                    continue
                pattern = rf'(?:^[-*\s]*)?{re.escape(pid)}\s*[:\-–—]\s*(.+)'
                match = re.search(pattern, line, re.IGNORECASE)
                if match:
                    ctx.cons[pid] = match.group(1).strip()
                    break
            if pid not in ctx.cons:
                ctx.cons[pid] = f"Priced at ${candidate['price']}"

        tracer.record("analyze_cons", {"candidates": [c["id"] for c in ctx.candidates]}, ctx.cons)
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        candidates = ctx.candidates[:]

        # Filter by max_price if set
        if ctx.max_price is not None:
            candidates = [c for c in candidates if c["price"] <= ctx.max_price]

        # Sort by highest rating, then lowest price as tiebreaker
        candidates.sort(key=lambda x: (-x["rating"], x["price"]))

        if candidates:
            ctx.best = candidates[0]

        tracer.record("rank_candidates", {
            "num_candidates": len(ctx.candidates),
            "max_price": ctx.max_price,
        }, ctx.best)
        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        ctx = AgentContext(query=user_message)
        tracer = ToolTracer()
        trace = []

        # 1. Retriever
        ctx = self.retriever.run(ctx, state, tools, tracer)
        trace.append("delegate_retriever")

        # 2. Pros analysis
        ctx = self.pros_agent.run(ctx, tracer)
        trace.append("delegate_pros")

        # 3. Cons analysis
        ctx = self.cons_agent.run(ctx, tracer)
        trace.append("delegate_cons")

        # 4. Ranking
        ctx = self.ranker.run(ctx, tracer)
        trace.append("delegate_ranker")

        # 5. Add to cart if requested
        if _wants_add_to_cart(user_message) and ctx.best:
            result = tools.add_to_cart(state, ctx.best["id"])
            ctx.cart_result = result
            tracer.record("add_to_cart", {"product_id": ctx.best["id"]}, result)
            trace.append("delegate_cart")

        # Build response
        if ctx.best:
            pid = ctx.best["id"]
            pros_text = ctx.pros.get(pid, "N/A")
            cons_text = ctx.cons.get(pid, "N/A")
            response = (
                f"Best recommendation: {ctx.best['name']}\n"
                f"Price: ${ctx.best['price']}\n"
                f"Rating: {ctx.best['rating']}\n"
                f"Pros: {pros_text}\n"
                f"Cons: {cons_text}"
            )
            if ctx.cart_result and ctx.cart_result.get("ok"):
                response += f"\n\nAdded to cart!"
        else:
            response = "No matching products found."

        return AgentResult(response=response, trace=trace, context=ctx)


In [ ]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


In [ ]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


In [ ]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")
